In [ ]:
# pip install -U pandas numpy scikit-learn matplotlib seaborn joblib
# If you're in a notebook environment, you may prefer:
# %pip install -U pandas numpy scikit-learn matplotlib seaborn joblib

%matplotlib inline


# Task 2 — ML Pipeline for Customer Churn (scikit-learn)

## Objective
Build a production-style churn classification pipeline with preprocessing, model comparison, hyperparameter tuning, and export.

## Dataset
IBM Telco Customer Churn dataset (public CSV):
https://raw.githubusercontent.com/IBM/telco-customer-churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv

## Approach
- Load data via a public URL (no login)
- Preprocess using `ColumnTransformer`:
  - `OneHotEncoder` for categorical columns
  - `StandardScaler` for numeric columns
- Compare Logistic Regression vs Random Forest
- Tune using `GridSearchCV` on a single pipeline
- Evaluate with classification report, ROC AUC, and confusion matrix
- Export best pipeline with `joblib.dump()` and reload for inference

## Final Summary / Insights
This notebook shows how to build an end-to-end sklearn pipeline that is easy to deploy (single `.pkl`) and robust to categorical feature handling.


In [ ]:
import os
import warnings
from typing import List

import numpy as np
import pandas as pd

from joblib import dump, load

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_PATH = "./task2_telco_churn_pipeline.pkl"


In [ ]:
# Load dataset (handle network errors gracefully)
try:
    df = pd.read_csv(DATA_URL)
except Exception as e:
    raise RuntimeError(
        "Failed to download/load the Telco Churn CSV. Check your internet connection and try again. "
        f"Original error: {e}"
    ) from e

print("Shape:", df.shape)
display(df.head())


In [ ]:
# Basic cleanup
# - TotalCharges can be loaded as string due to blanks
df = df.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Target
y = (df["Churn"].astype(str).str.strip() == "Yes").astype(int)

# Features (drop ID + target)
X = df.drop(columns=["customerID", "Churn"])

categorical_cols: List[str] = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols: List[str] = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
# Preprocessing pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# One pipeline, multiple models via GridSearchCV
pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=2000, solver="lbfgs")),
    ]
)

param_grid = [
    {
        "clf": [LogisticRegression(max_iter=2000, solver="lbfgs")],
        "clf__C": [0.1, 1.0, 3.0],
    },
    {
        "clf": [RandomForestClassifier(random_state=42)],
        "clf__n_estimators": [200, 500],
        "clf__max_depth": [None, 8, 16],
        "clf__min_samples_split": [2, 10],
    },
]

search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_model = search.best_estimator_


In [ ]:
# Evaluate on held-out test set
y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["No", "Yes"]))
test_auc = roc_auc_score(y_test, y_proba)
print("Test ROC AUC:", test_auc)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve")
plt.show()


In [ ]:
# Export and reload
dump(best_model, MODEL_PATH)
print("Saved best pipeline to:", MODEL_PATH)

reloaded = load(MODEL_PATH)
print("Reloaded pipeline:", type(reloaded))

# Predict on a few new rows (sample from test set)
sample_X = X_test.sample(n=3, random_state=42)
sample_proba = reloaded.predict_proba(sample_X)[:, 1]

out = sample_X.copy()
out["churn_probability"] = sample_proba
display(out)
